In [49]:
!pip install tensorflow -q

In [50]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [51]:
data={
     "Training_Hours": [
        2, 3, 4, 5, 6,
        7, 8, 9, 10, 11,
        12, 13, 14, 15, 16
    ],

    "Attendance": [
        60, 65, 62, 68, 70,
        72, 75, 78, 80, 82,
        85, 86, 88, 90, 92
    ],

    "Performance": [
        "Needs Improvement",
        "Needs Improvement",
        "Needs Improvement",
        "Needs Improvement",
        "Needs Improvement",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good",
        "Good"
    ]
}
df=pd.DataFrame(data)
display(df)

,Training_Hours,Attendance,Performance
0,2,60,Needs Improvement
1,3,65,Needs Improvement
2,4,62,Needs Improvement
3,5,68,Needs Improvement
4,6,70,Needs Improvement
5,7,72,Good
6,8,75,Good
7,9,78,Good
8,10,80,Good
9,11,82,Good


In [52]:
x=df[["Training_Hours","Attendance"]]
y=df["Performance"]

In [73]:
encoder=LabelEncoder()
y=encoder.fit_transform(y)
print("encoded result")
y=1-y
print(y)

encoded result
[0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]


In [74]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print("Training records:", len(x_train))
print("Testing records:", len(x_test))

Training records: 12
Testing records: 3


In [75]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        8,
        activation="relu",
        input_shape=(2,)
    ),
    tf.keras.layers.Dense(
        4,
        activation="relu"
    ),
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [76]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65 (260.00 B)

 Trainable params: 65 (260.00 B)

 Non-trainable params: 0 (0.00 B)

In [77]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [78]:
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    verbose=0
)

In [79]:
print("ANN training completed!")

ANN training completed!


In [82]:
probability = model.predict(x_test, verbose=0)

In [83]:
y_pred = (probability >= 0.5).astype(int).flatten()

In [84]:
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 1.0


In [85]:
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 100.0 %


In [86]:
result=pd.DataFrame({
    "Training_Hours":x_test["Training_Hours"].values,
    "Attendance":x_test["Attendance"].values,
    "performance":y_test,
    "pred_Performance":y_pred
})
display(result)

,Training_Hours,Attendance,performance,pred_Performance
0,11,82,1,1
1,13,86,1,1
2,2,60,0,0


In [87]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Needs Improvement","Good"], zero_division=0))

Classification Report:
                   precision    recall  f1-score   support

Needs Improvement       1.00      1.00      1.00         1
             Good       1.00      1.00      1.00         2

         accuracy                           1.00         3
        macro avg       1.00      1.00      1.00         3
     weighted avg       1.00      1.00      1.00         3



In [88]:
new_data=pd.DataFrame({
      "Training_Hours": [
        4,6,9,12,15
    ],

    "Attendance": [
       65,72,80,87,91
    ]
})
new_data

,Training_Hours,Attendance
0,4,65
1,6,72
2,9,80
3,12,87
4,15,91


In [90]:
new_probability=model.predict(new_data,verbose=0)

In [91]:
new_prediction = (new_probability >= 0.5).astype(int).flatten()

In [92]:
print("new prediction")
res=[]
for i in range(len(new_data)):
  if new_prediction[i]==1:
    result_text="Good"
  else:
    result_text="Need Improvement"
  res.append({
      "Training_Hours":new_data["Training_Hours"][i],
      "Attendance":new_data["Attendance"][i],
      "probability": str(round(float(new_probability[i][0]) * 100, 2)) + "%",
      "Performance":result_text
  })
  result_df=pd.DataFrame(res)
result_df

new prediction


,Training_Hours,Attendance,probability,Performance
0,4,65,25.99%,Need Improvement
1,6,72,45.1%,Need Improvement
2,9,80,77.1%,Good
3,12,87,93.58%,Good
4,15,91,98.08%,Good


In [93]:
model.save("ANN_emp_pred_performance.keras")
print("model saved")

model saved
